Code to sum up the ColonSigmoid and ColonTransverse columns to a new column called Colon for the reference matrix. Esophagus and EsophagusMucosa are joined as the input samples just states Esophagus.
This code also scales the tissue proportions to 100% for the different tools that their output is sum to 1 (CIBERSORTx, MuSiC and BayesPrism)

In [1]:
import os
import pandas as pd

# Define the root directory
root_dir = 'TOO-Decon-Degradation'

# Helper function to normalize the row sums to 100
def normalize_to_100(df):
    # Assume the first column is sample names
    sample_col = df.columns[0]
    # Columns to normalize: all numeric ones excluding the first column
    columns_to_normalize = [col for col in df.columns if col != sample_col and pd.api.types.is_numeric_dtype(df[col])]    
    # Normalize each row to a sum of 100
    df[columns_to_normalize] = df[columns_to_normalize].div(df[columns_to_normalize].sum(axis=1), axis=0) * 100
    return df

# Column renaming mapping
column_rename_map = {
    'Whole_blood': 'Whole blood',
    'Adrenal_gland': 'Adrenal gland'
}

# Loop through each subdirectory and file
for subdir, dirs, files in os.walk(root_dir):
    if 'Decon-Results_' in subdir:
        for file in files:
            if file.endswith('.txt') and '_modified.txt' not in file and '_RMSE.txt' not in file:
                file_path = os.path.join(subdir, file)
                
                try:
                    df = pd.read_csv(file_path, sep="\t", index_col= 0)

                    if 'ColonSigmoid' in df.columns and 'ColonTransverse' in df.columns:
                        df['Colon'] = df['ColonSigmoid'] + df['ColonTransverse']
                        df['Esophagus'] = df['Esophagus'] + df['EsophagusMucosa']
                        df = df.drop(columns=['ColonSigmoid', 'ColonTransverse', 'EsophagusMucosa'])

                        # Harmonize column names and range of proportions (normalize all to reach 100) for specific files
                        if file.endswith('BayesPrism.txt') or file.endswith('MuSiC_proportions.txt'):
                            df = df.rename(columns=column_rename_map)
                            df.index = df.index.str.replace('.', '-', regex=False)  # Replace '.' with '-' in row index

                        if file.endswith('BayesPrism.txt') or file.endswith('MuSiC_proportions.txt') or file == 'CIBERSORTx_Results.txt' or file.endswith('ReDeconv_results.txt'):
                            if file == 'CIBERSORTx_Results.txt':
                                columns_to_exclude = ['P-value', 'Correlation', 'RMSE']
                                df = df.drop(columns=columns_to_exclude, errors='ignore')

                            df = normalize_to_100(df)

                        # Round all numeric values to 5 decimal places before saving
                        df = df.round(5)
                
                        output_file_path = file_path.replace('.txt', '_modified.txt')
                        df.to_csv(output_file_path, sep='\t')
                        print(f"Processed and saved: {output_file_path}")
                    else:
                        print(f"Columns 'ColonSigmoid' and 'ColonTransverse' not found in {file_path}")
                
                except Exception as e:
                    print(f"Error processing {file_path}: {e}")


Processed and saved: TOO-Decon-Degradation/Decon-Results_20250616_All-Tissues-NoDup_Random_Degradation_top_30_percent_removed_2Median_300_500/20250405_GeneID_LessTissueV2_2Median-withGTFNames_20250616_All-Tissues-NoDup_Random_Degradation_top_30_percent_removed_BayesPrism_modified.txt
Processed and saved: TOO-Decon-Degradation/Decon-Results_20250616_All-Tissues-NoDup_Random_Degradation_top_30_percent_removed_2Median_300_500/nuSVR_CountsRMSE_20250616_All-Tissues-NoDup_Random_Degradation_top_30_percent_removed_modified.txt
Processed and saved: TOO-Decon-Degradation/Decon-Results_20250616_All-Tissues-NoDup_Random_Degradation_top_30_percent_removed_2Median_300_500/NNLS_20250616_All-Tissues-NoDup_Random_Degradation_top_30_percent_removed_modified.txt
Processed and saved: TOO-Decon-Degradation/Decon-Results_20250616_All-Tissues-NoDup_Random_Degradation_top_30_percent_removed_2Median_300_500/20250405_GeneID_LessTissueV2_2Median-withGTFNames__20250616_All-Tissues-NoDup_Random_Degradation_top_30

# Deconvolution Results RMSE Evaluation

This script compares estimated tissue proportions from deconvolution methods to known ground truth and computes the RMSE (Root Mean Squared Error) for each modified result file.

## Workflow Overview

1. **Input Files**:
   - Known proportions file with tissue proportions per sample.
   - Deconvolution result files ending with `_modified.txt` under `Decon-Results_*` directories.

2. **Proportions Preprocessing**:
   - Combines `Cervix`, `Uterus`, and `Ovary` into `FemaleReproductive`.
   - Renames columns for consistency:
     - `Thyroid-gland` → `Thyroid`
     - `Adrenal-gland` → `Adrenal_gland`
     - `Small-Intestine` → `SmallIntestine`
     - `Skeletal-muscle` → `MuscleSkeletal`
   - Adds missing tissues (present in deconvolution but not in proportions) with zero values.
   - Ensures all files share the same tissue order.

3. **Deconvolution Results Comparison**:
   - Merges each deconvolution file with the preprocessed proportions by `Sample`.
   - Calculates RMSE, MAE and Pearson correlation between known and estimated tissue proportions per sample.
   - Saves it as '*_Metrics.txt' file in the same directory

In [2]:
## For the Random samples:

import os
import pandas as pd
import numpy as np
from glob import glob
from scipy.stats import pearsonr

# === CONFIGURATION ===
proportions_file = 'TOO-Decon-Degradation/Data/20250616_All-Tissues-NoDup_Random_Simulated_v2_Proportions.txt'
root_dir = 'TOO-Decon-Degradation'

# === LOAD AND PREPARE PROPORTIONS FILE ===
prop_df = pd.read_csv(proportions_file, sep='\t', index_col=0)

# Merge and rename columns to match reference matrix
prop_df['FemaleReproductive'] = prop_df[['Cervix', 'Ovary', 'Uterus']].sum(axis=1)
prop_df = prop_df.rename(columns={
    'Thyroid-gland': 'Thyroid',
    'Adrenal-gland': 'Adrenal gland',
    'Small-Intestine': 'SmallIntestine',
    'Skeletal-muscle': 'MuscleSkeletal'
})
prop_df = prop_df.drop(columns=['Cervix', 'Ovary', 'Uterus', 'Thyroid-gland', 'Adrenal-gland', 'Small-Intestine', 'Skeletal-muscle'], errors='ignore')

# List of all reference tissues
reference_tissues = [
    'Adipose', 'Adrenal gland', 'Arteries', 'Bladder', 'Brain', 'Breast', 'Colon', 'Esophagus', 'FemaleReproductive',
    'Fibroblasts', 'Heart', 'Kidney', 'Liver', 'Lung', 'Lymphocytes', 'MuscleSkeletal', 'NerveTibial', 'Pancreas',
    'Pituitary', 'Prostate', 'SalivaryGland', 'Skin', 'SmallIntestine', 'Spleen', 'Stomach', 'Testis', 'Thyroid', 'Whole blood'
]


# Add missing columns as zero and normalize
for tissue in reference_tissues:
    if tissue not in prop_df.columns:
        prop_df[tissue] = 0.0

prop_df_tissues = prop_df[reference_tissues].copy()
prop_df_tissues = prop_df_tissues.div(prop_df_tissues.sum(axis=1), axis=0).multiply(100).round(5)
prop_df.update(prop_df_tissues)

# === METRIC FUNCTIONS ===
def calculate_rmse(y_true, y_pred):
    return np.sqrt(((y_true - y_pred) ** 2).mean())

def calculate_mae(y_true, y_pred):
    return np.abs(y_true - y_pred).mean()

def calculate_pearson(y_true, y_pred):
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan  # Avoid division by zero in correlation
    return pearsonr(y_true, y_pred)[0]

# === PROCESS DECON FILES ===
decon_dirs = sorted(glob(os.path.join(root_dir, 'Decon-Results_*Random_*_500')))

for dir_path in decon_dirs:
    modified_files = sorted(glob(os.path.join(dir_path, '*_modified.txt')))
    
    for file_path in modified_files:
        try:
            decon_df = pd.read_csv(file_path, sep='\t', index_col=0)

            # Ensure all reference columns exist
            for col in reference_tissues:
                if col not in decon_df.columns:
                    decon_df[col] = 0.0
            decon_df = decon_df[reference_tissues]

            # Initialize metrics
            rmse_scores = {}
            mae_scores = {}
            pearson_scores = {}

            # Match and compute metrics per sample
            common_samples = prop_df.index.intersection(decon_df.index)
            for sample in common_samples:
                true_vals = prop_df.loc[sample]
                pred_vals = decon_df.loc[sample]

                rmse_scores[sample] = calculate_rmse(true_vals, pred_vals)
                mae_scores[sample] = calculate_mae(true_vals, pred_vals)
                pearson_scores[sample] = calculate_pearson(true_vals, pred_vals)

            # Add metric columns
            decon_df['RMSE-Composition'] = pd.Series(rmse_scores)
            decon_df['MAE-Composition'] = pd.Series(mae_scores)
            decon_df['Pearson-Composition'] = pd.Series(pearson_scores)

            # Save results
            out_path = file_path.replace('_modified.txt', '_Metrics.txt')
            decon_df.to_csv(out_path, sep='\t')
            print(f"Saved: {out_path}")

        except Exception as e:
            print(f"Error processing {file_path}: {e}")


Saved: TOO-Decon-Degradation/Decon-Results_20250616_All-Tissues-NoDup_Random_Degradation_top_0_percent_removed_2Median_300_500/20250405_GeneID_LessTissueV2_2Median-withGTFNames_20250616_All-Tissues-NoDup_Random_Simulated_v2_Counts_BayesPrism_Metrics.txt
Saved: TOO-Decon-Degradation/Decon-Results_20250616_All-Tissues-NoDup_Random_Degradation_top_0_percent_removed_2Median_300_500/20250405_GeneID_LessTissueV2_2Median-withGTFNames_Random_Simulated_v2_MuSiC_proportions_Metrics.txt
Saved: TOO-Decon-Degradation/Decon-Results_20250616_All-Tissues-NoDup_Random_Degradation_top_0_percent_removed_2Median_300_500/20250405_GeneID_LessTissueV2_Sampling10-Unique_Top500_ReDeconv_results_Metrics.txt
Saved: TOO-Decon-Degradation/Decon-Results_20250616_All-Tissues-NoDup_Random_Degradation_top_0_percent_removed_2Median_300_500/CIBERSORTx_Results_Metrics.txt
Saved: TOO-Decon-Degradation/Decon-Results_20250616_All-Tissues-NoDup_Random_Degradation_top_0_percent_removed_2Median_300_500/NNLS_20250616_All-Tissue

In [3]:
import os
import pandas as pd
from glob import glob

# === CONFIGURATION ===
root_dir = 'TOO-Decon-Degradation'
method_map = ['CIBERSORTx', 'MuSiC', 'QP', 'NNLS', 'BayesPrism', 'nuSVR', 'ReDeconv']

# === GATHER METRICS FILES ===
uniform_dirs = sorted(glob(os.path.join(root_dir, 'Decon-Results_*Random_*_500')))
metrics_data = []

# === EXTRACT METRIC VALUES ===
for dir_path in uniform_dirs:
    metrics_files = glob(os.path.join(dir_path, '*_Metrics.txt'))  # Adjusted from _RMSE to _Metrics

    for file_path in metrics_files:
        try:
            df = pd.read_csv(file_path, sep='\t', index_col=0)

            # Check for required metric columns
            if all(col in df.columns for col in ['RMSE-Composition', 'MAE-Composition', 'Pearson-Composition']) and not df.empty:
                
                # Extract method name
                raw_method = os.path.splitext(os.path.basename(file_path))[0].replace('_Metrics', '')
                matched_method = next((m for m in method_map if m in raw_method), raw_method)

                # Extract directory parts
                full_dir_name = os.path.basename(dir_path)
                dir_parts = full_dir_name.replace('Decon-Results_', '').split('_')

                # Extract matrix (last 3 parts)
                matrix = '_'.join(dir_parts[-3:]) if len(dir_parts) >= 3 else full_dir_name

                # Extract gene removal (e.g., 'top_10')
                gene_removal = 'NA'
                for i in range(len(dir_parts) - 1):
                    if dir_parts[i] == 'top' and dir_parts[i + 1].isdigit():
                        gene_removal = f'top_{dir_parts[i + 1]}'
                        break

                # Collect metrics for each sample
                for sample_name, row in df.iterrows():
                    metrics_data.append({
                        'Sample': sample_name,
                        'Method': matched_method,
                        'Matrix': matrix,
                        'GeneRemoval': gene_removal,
                        'RMSE-Composition': row['RMSE-Composition'],
                        'MAE-Composition': row['MAE-Composition'],
                        'Pearson-Composition': row['Pearson-Composition']
                    })

        except Exception as e:
            print(f"Error reading {file_path}: {e}")

# === CREATE MERGED DATAFRAME ===
metrics_df = pd.DataFrame(metrics_data)

# === SAVE TO FILE ===
output_path = 'merged_Results_Metrics_Random_TOOv2.txt'
metrics_df.to_csv(output_path, sep='\t', index=False)
print(f"Saved merged metrics: {output_path}")


Saved merged metrics: merged_Results_Metrics_Random_TOOv2.txt


In [4]:
## Extract the spillover per tissue type by obtaining the absolute difference of the computed proportion vs the ground truth
## For the Random samples:

import os
import pandas as pd
import numpy as np
from glob import glob
from scipy.stats import pearsonr

# === CONFIGURATION ===
proportions_file = 'TOO-Decon-Degradation/Data/20250616_All-Tissues-NoDup_Random_Simulated_v2_Proportions.txt'
root_dir = 'TOO-Decon-Degradation'

# === LOAD AND PREPARE PROPORTIONS FILE ===
prop_df = pd.read_csv(proportions_file, sep='\t', index_col=0)

# Merge and rename columns to match reference matrix
prop_df['FemaleReproductive'] = prop_df[['Cervix', 'Ovary', 'Uterus']].sum(axis=1)
prop_df = prop_df.rename(columns={
    'Thyroid-gland': 'Thyroid',
    'Adrenal-gland': 'Adrenal gland',
    'Small-Intestine': 'SmallIntestine',
    'Skeletal-muscle': 'MuscleSkeletal'
})
prop_df = prop_df.drop(columns=['Cervix', 'Ovary', 'Uterus', 'Thyroid-gland', 'Adrenal-gland', 'Small-Intestine', 'Skeletal-muscle'], errors='ignore')

# List of all reference tissues
reference_tissues = [
    'Adipose', 'Adrenal gland', 'Arteries', 'Bladder', 'Brain', 'Breast', 'Colon', 'Esophagus', 'FemaleReproductive',
    'Fibroblasts', 'Heart', 'Kidney', 'Liver', 'Lung', 'Lymphocytes', 'MuscleSkeletal', 'NerveTibial', 'Pancreas',
    'Pituitary', 'Prostate', 'SalivaryGland', 'Skin', 'SmallIntestine', 'Spleen', 'Stomach', 'Testis', 'Thyroid', 'Whole blood'
]


# Add missing columns as zero and normalize
for tissue in reference_tissues:
    if tissue not in prop_df.columns:
        prop_df[tissue] = 0.0

prop_df_tissues = prop_df[reference_tissues].copy()
prop_df_tissues = prop_df_tissues.div(prop_df_tissues.sum(axis=1), axis=0).multiply(100).round(5)
prop_df.update(prop_df_tissues)

# === PROCESS DECON FILES ===
decon_dirs = sorted(glob(os.path.join(root_dir, 'Decon-Results_*Random_*_500')))

for dir_path in decon_dirs:
    modified_files = sorted(glob(os.path.join(dir_path, '*_modified.txt')))
    
    for file_path in modified_files:
        try:
            decon_df = pd.read_csv(file_path, sep='\t', index_col=0)

            # Ensure all reference columns exist
            for col in reference_tissues:
                if col not in decon_df.columns:
                    decon_df[col] = 0.0
            decon_df = decon_df[reference_tissues]

            # Subset to samples that exist in both
            common_samples = prop_df.index.intersection(decon_df.index)

            # Subset and align
            true_vals_df = prop_df.loc[common_samples, reference_tissues]
            pred_vals_df = decon_df.loc[common_samples, reference_tissues]

            # Compute absolute difference (spillover)
            spillover_df = (pred_vals_df - true_vals_df).abs()

            # Save spillover
            out_path = file_path.replace('_modified.txt', '_Spillover.txt')
            spillover_df.to_csv(out_path, sep='\t')
            print(f"Saved spillover: {out_path}")

        except Exception as e:
            print(f"Error processing {file_path}: {e}")


Saved spillover: TOO-Decon-Degradation/Decon-Results_20250616_All-Tissues-NoDup_Random_Degradation_top_0_percent_removed_2Median_300_500/20250405_GeneID_LessTissueV2_2Median-withGTFNames_20250616_All-Tissues-NoDup_Random_Simulated_v2_Counts_BayesPrism_Spillover.txt
Saved spillover: TOO-Decon-Degradation/Decon-Results_20250616_All-Tissues-NoDup_Random_Degradation_top_0_percent_removed_2Median_300_500/20250405_GeneID_LessTissueV2_2Median-withGTFNames_Random_Simulated_v2_MuSiC_proportions_Spillover.txt
Saved spillover: TOO-Decon-Degradation/Decon-Results_20250616_All-Tissues-NoDup_Random_Degradation_top_0_percent_removed_2Median_300_500/20250405_GeneID_LessTissueV2_Sampling10-Unique_Top500_ReDeconv_results_Spillover.txt
Saved spillover: TOO-Decon-Degradation/Decon-Results_20250616_All-Tissues-NoDup_Random_Degradation_top_0_percent_removed_2Median_300_500/CIBERSORTx_Results_Spillover.txt
Saved spillover: TOO-Decon-Degradation/Decon-Results_20250616_All-Tissues-NoDup_Random_Degradation_top_

In [5]:
# This script collects absolute differences (spillover) from all *_Spillover.txt files
# in the Random TOOv2 deconvolution results directories.
# It merges them into a dataframe with Sample, Method, Matrix, Tissue-specific Spillover values,
# and a new column 'Average' representing the mean spillover per sample.

import os
import pandas as pd
from glob import glob

# === CONFIGURATION ===
root_dir = 'TOO-Decon-Degradation'
method_map = ['CIBERSORTx', 'MuSiC', 'QP', 'NNLS', 'BayesPrism', 'nuSVR', 'ReDeconv']

# === GATHER SPILLOVER FILES ===
random_dirs = sorted(glob(os.path.join(root_dir, 'Decon-Results_*Random_*_500')))
spillover_data = []

# === PROCESS EACH SPILLOVER FILE ===
for dir_path in random_dirs:
    spillover_files = glob(os.path.join(dir_path, '*_Spillover.txt'))

    for file_path in spillover_files:
        try:
            # Load spillover values
            df = pd.read_csv(file_path, sep='\t', index_col=0)

            # Compute average spillover per sample
            df['Average'] = df.mean(axis=1)

            # Extract method name from file name
            filename = os.path.basename(file_path)
            raw_method = filename.replace('_Spillover.txt', '')
            method = next((m for m in method_map if m in raw_method), raw_method)

            # Extract matrix name from directory
            dir_name = os.path.basename(dir_path)
            parts = dir_name.split('_')

            # Matrix: last 3 parts joined
            matrix = '_'.join(parts[-3:])  # e.g., 2Median_1000_1500

            # GeneRemoval: detect the full "top_10" (e.g., top + next number)
            gene_removal = 'NA'
            for i, p in enumerate(parts):
                    if p == 'top' and i + 1 < len(parts):
                        gene_removal = f'top_{parts[i+1]}'
                        break

            # Add metadata
            df['Method'] = method
            df['Matrix'] = matrix
            df['GeneRemoval'] = gene_removal
            df['Sample'] = df.index

            spillover_data.append(df.reset_index(drop=True))

        except Exception as e:
            print(f"Error reading {file_path}: {e}")

# === MERGE ALL RESULTS ===
merged_df = pd.concat(spillover_data, ignore_index=True)

# === SAVE TO FILE ===
output_path = 'merged_Results_Spillover_Random_TOOv2.txt'
merged_df.to_csv(output_path, sep='\t', index=False)
print(f"Saved merged spillover results: {output_path}")

Saved merged spillover results: merged_Results_Spillover_Random_TOOv2.txt
